In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


# Thu thập Comment từ Link Video / Bài viết TikTok

Notebook này mở trình duyệt **Microsoft Edge** có sẵn trên Windows (`channel="msedge"`, `headless=False`) bằng **Playwright**, hỗ trợ link video (`/video/...`), bài viết ảnh (`/photo/...`) hoặc link rút gọn (`vt.tiktok.com/...`), bắt gói dữ liệu bình luận và xuất ra file `.csv` tương thích trực tiếp với `phase1.ipynb`.

### Chuẩn bị (chỉ cần làm 1 lần)
Chỉ cần cài thư viện `playwright`, không cần tải thêm Chromium vì đã dùng Microsoft Edge sẵn có trên máy:
```powershell
pip install playwright
```

### Lưu ý khi chạy
- Sau khi trình duyệt mở trang TikTok, notebook sẽ **tạm dừng** để bạn tự đóng popup đăng nhập, chuyển sang tab bình luận (nếu cần), hoặc tự hoàn tất bước xác minh của TikTok trực tiếp trên cửa sổ trình duyệt.
- Khi đã thấy khu vực bình luận sẵn sàng, quay lại đây nhấn **Enter** để script cuộn và thu thập tự động.
- Cột đầu tiên của file `.csv` xuất ra là `comment` (chuẩn `utf-8-sig`), dùng thẳng cho biến `COMMENTS_PATH` trong `phase1.ipynb`.

In [2]:
# (Tùy chọn) Bỏ dấu # nếu môi trường hiện tại chưa cài Playwright:
%pip install playwright

Note: you may need to restart the kernel to use updated packages.


In [3]:
n = input("Nhập link video hoặc bài viết ảnh TikTok: ")

In [4]:
# === CẤU HÌNH THU THẬP COMMENT ===
# Dán link video hoặc bài viết ảnh TikTok của bạn vào đây (hỗ trợ cả vt.tiktok.com/...)
TIKTOK_URL = n

MAX_COMMENTS = 200       # Số lượng bình luận tối đa muốn lấy
MAX_SCROLLS = 40         # Số lần cuộn tối đa
SCROLL_PAUSE_SEC = 5.0   # Thời gian chờ (giây) giữa mỗi lần cuộn để tải thêm comment
INCLUDE_REPLIES = False  # True nếu muốn thu thập thêm cả phản hồi (reply) khi bạn mở chúng
BROWSER_CHANNEL = "msedge"  # Dùng Microsoft Edge sẵn có trên Windows ("msedge" / "chrome" / None)

# Để None để tự động lưu vào data/tiktok_comments_<post_id>.csv
OUTPUT_CSV = None

In [5]:
import concurrent.futures
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


def extract_post_id(url: str) -> str:
    """Trích xuất ID bài viết từ URL TikTok (/video/... hoặc /photo/...)."""
    match = re.search(r"/(?:video|photo)/(\d+)", str(url))
    if match:
        return match.group(1)
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def _format_unix_ts(ts_value) -> str:
    if not ts_value:
        return ""
    try:
        dt = datetime.fromtimestamp(int(ts_value), tz=timezone.utc).astimezone()
        return dt.strftime("%Y-%m-%d %H:%M:%S")
    except (ValueError, TypeError, OSError):
        return ""


def _parse_comment_items(payload: dict, post_url: str) -> list[dict]:
    """Chuyển JSON response của API bình luận TikTok thành danh sách dict chuẩn."""
    if not isinstance(payload, dict):
        return []

    raw_comments = payload.get("comments") or []
    if not isinstance(raw_comments, list):
        return []

    rows: list[dict] = []
    for item in raw_comments:
        if not isinstance(item, dict):
            continue
        text = str(item.get("text") or "").strip()
        if not text:
            continue

        user = item.get("user") if isinstance(item.get("user"), dict) else {}
        cid = str(item.get("cid") or "").strip()
        rows.append(
            {
                "comment": text,
                "comment_id": cid,
                "username": str(user.get("unique_id") or "").strip(),
                "nickname": str(user.get("nickname") or "").strip(),
                "likes": int(item.get("digg_count") or 0),
                "reply_count": int(item.get("reply_comment_total") or 0),
                "created_at": _format_unix_ts(item.get("create_time")),
                "post_url": post_url,
            }
        )
    return rows


def _extract_dom_fallback(page, post_url: str) -> list[dict]:
    """Dự phòng đọc bình luận đã render sẵn trên DOM nếu response đầu tiên xảy ra trước khi gắn listener."""
    rows: list[dict] = []
    selectors = [
        'div[data-e2e="comment-level-1"]',
        '[class*="DivCommentItemContainer"]',
    ]
    elements = []
    for sel in selectors:
        found = page.locator(sel).all()
        if found:
            elements = found
            break

    for el in elements:
        try:
            text_loc = el.locator('[data-e2e="comment-level-1"] span, span[data-e2e="comment-text"], p[data-e2e="comment-level-1"]').first
            text = text_loc.inner_text(timeout=500).strip()
        except Exception:
            try:
                text = el.inner_text(timeout=500).strip()
            except Exception:
                text = ""
        if not text:
            continue

        username = ""
        try:
            username = el.locator('[data-e2e="comment-username-1"], span[data-e2e="comment-username"]').first.inner_text(timeout=300).strip()
        except Exception:
            pass

        rows.append(
            {
                "comment": text,
                "comment_id": "",
                "username": username,
                "nickname": "",
                "likes": 0,
                "reply_count": 0,
                "created_at": "",
                "post_url": post_url,
            }
        )
    return rows


def _collect_sync(
    url: str,
    max_comments: int = 200,
    max_scrolls: int = 30,
    scroll_pause_sec: float = 2.0,
    include_replies: bool = False,
    browser_channel: str | None = "msedge",
) -> tuple[list[dict], str]:
    """Chạy Playwright Sync API trong luồng riêng để tương thích hoàn toàn với Jupyter trên Windows."""
    try:
        from playwright.sync_api import sync_playwright
    except ImportError as exc:
        raise ImportError(
            "Chưa cài đặt `playwright`. Hãy chạy: pip install playwright"
        ) from exc

    collected: dict[str, dict] = {}
    resolved_url = url

    def add_rows(rows: list[dict]) -> int:
        added = 0
        for row in rows:
            dedup_key = row["comment_id"] or f"{row['username']}::{row['comment']}"
            if dedup_key not in collected:
                collected[dedup_key] = row
                added += 1
        return added

    with sync_playwright() as p:
        launch_kwargs: dict = {"headless": False}
        if browser_channel:
            launch_kwargs["channel"] = browser_channel
        browser = p.chromium.launch(**launch_kwargs)
        context = browser.new_context(
            viewport={"width": 1280, "height": 860},
            locale="vi-VN",
        )
        page = context.new_page()

        def handle_response(response) -> None:
            resp_url = response.url
            is_main = "/api/comment/list/" in resp_url and "/api/comment/list/reply/" not in resp_url
            is_reply = "/api/comment/list/reply/" in resp_url
            if not (is_main or (include_replies and is_reply)):
                return
            if response.status != 200:
                return
            try:
                payload = response.json()
                add_rows(_parse_comment_items(payload, resolved_url))
            except Exception:
                pass

        page.on("response", handle_response)

        print(f"Đang mở trình duyệt tới: {url}")
        page.goto(url, wait_until="domcontentloaded", timeout=60000)
        time.sleep(3.0)
        resolved_url = page.url

        print("\n[Tạm dừng] Trên cửa sổ Microsoft Edge vừa mở:")
        print("  1. Đóng popup đăng nhập (hoặc đăng nhập nếu TikTok yêu cầu).")
        print("  2. Mở phần bình luận (nếu chưa hiện) hoặc hoàn tất bước xác minh (nếu có).")
        input("Khi khu vực bình luận đã hiển thị sẵn sàng, nhấn Enter tại đây để bắt đầu thu thập...")

        resolved_url = page.url
        add_rows(_extract_dom_fallback(page, resolved_url))

        idle_scrolls = 0
        for step in range(1, max_scrolls + 1):
            if len(collected) >= max_comments:
                print(f"Đã đạt giới hạn {max_comments} bình luận.")
                break

            before_count = len(collected)

            # 1. Cuộn trực tiếp tới comment cuối cùng đang hiển thị để kích hoạt tải thêm
            for comment_sel in (
                'div[data-e2e="comment-level-1"]',
                '[class*="DivCommentItemContainer"]',
                '[class*="DivCommentListContainer"] > div',
            ):
                try:
                    locs = page.locator(comment_sel)
                    cnt = locs.count()
                    if cnt > 0:
                        locs.nth(cnt - 1).scroll_into_view_if_needed(timeout=1000)
                        break
                except Exception:
                    pass

            # 2. Cuộn container có thanh cuộn bao quanh danh sách comment + cuộn cửa sổ trang
            try:
                page.evaluate(
                    """() => {
                        const el = document.querySelector('[class*="DivCommentListContainer"], div[data-e2e="comment-level-1"]');
                        let node = el ? el.parentElement : null;
                        while (node && node !== document.body) {
                            const style = window.getComputedStyle(node);
                            const overflowY = style.overflowY;
                            if ((overflowY === 'auto' || overflowY === 'scroll') && node.scrollHeight > node.clientHeight + 20) {
                                node.scrollTop += 1400;
                                break;
                            }
                            node = node.parentElement;
                        }
                        window.scrollBy(0, 1400);
                    }"""
                )
            except Exception:
                pass

            # 3. Dự phòng bằng phím PageDown
            page.keyboard.press("PageDown")
            time.sleep(scroll_pause_sec)

            # Đọc bổ sung DOM nếu gói mạng chưa kịp khớp hoặc comment mới vừa render
            if len(collected) == before_count:
                add_rows(_extract_dom_fallback(page, resolved_url))

            after_count = len(collected)
            print(f"  Lần cuộn {step}/{max_scrolls}: thu được {after_count} bình luận")

            if after_count == before_count:
                idle_scrolls += 1
                if idle_scrolls >= 4:
                    print("Không tải thêm được bình luận mới sau 4 lần cuộn liên tiếp, dừng thu thập.")
                    if after_count <= 20:
                        print("[Gợi ý] TikTok thường giới hạn chỉ hiện 10-20 bình luận đầu nếu chưa đăng nhập trên cửa sổ Edge đó.")
                    break
            else:
                idle_scrolls = 0

        browser.close()

    rows = list(collected.values())[:max_comments]
    return rows, resolved_url


def collect_tiktok_comments(
    url: str,
    max_comments: int = 200,
    max_scrolls: int = 30,
    scroll_pause_sec: float = 2.0,
    include_replies: bool = False,
    browser_channel: str | None = "msedge",
) -> tuple[list[dict], str]:
    """Bọc gọi _collect_sync trong ThreadPoolExecutor để tránh xung đột event loop của Jupyter Windows."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(
            _collect_sync,
            url,
            max_comments,
            max_scrolls,
            scroll_pause_sec,
            include_replies,
            browser_channel,
        )
        return future.result()

In [6]:
# === CHẠY THU THẬP COMMENT ===
if not TIKTOK_URL or "@username/video/1234567890123456789" in TIKTOK_URL:
    raise ValueError("Hãy thay TIKTOK_URL ở cell cấu hình bằng link video/bài viết TikTok thực tế trước khi chạy!")

rows, resolved_url = collect_tiktok_comments(
    url=TIKTOK_URL,
    max_comments=MAX_COMMENTS,
    max_scrolls=MAX_SCROLLS,
    scroll_pause_sec=SCROLL_PAUSE_SEC,
    include_replies=INCLUDE_REPLIES,
    browser_channel=BROWSER_CHANNEL,
)

df_comments = pd.DataFrame(rows)
print(f"\nTổng số bình luận thu được: {len(df_comments)}")
print(f"URL thực tế sau điều hướng: {resolved_url}")
df_comments.head(15)

Đang mở trình duyệt tới: https://www.tiktok.com/@phamlocvt/video/7708245390318996737?q=drama%20tr%C6%B0%E1%BB%9Dng%20giang&t=1795274259548

[Tạm dừng] Trên cửa sổ Microsoft Edge vừa mở:
  1. Đóng popup đăng nhập (hoặc đăng nhập nếu TikTok yêu cầu).
  2. Mở phần bình luận (nếu chưa hiện) hoặc hoàn tất bước xác minh (nếu có).
  Lần cuộn 1/40: thu được 46 bình luận
  Lần cuộn 2/40: thu được 66 bình luận
  Lần cuộn 3/40: thu được 88 bình luận
  Lần cuộn 4/40: thu được 109 bình luận
  Lần cuộn 5/40: thu được 128 bình luận
  Lần cuộn 6/40: thu được 148 bình luận
  Lần cuộn 7/40: thu được 170 bình luận
  Lần cuộn 8/40: thu được 193 bình luận
  Lần cuộn 9/40: thu được 198 bình luận
  Lần cuộn 10/40: thu được 198 bình luận
  Lần cuộn 11/40: thu được 198 bình luận
  Lần cuộn 12/40: thu được 198 bình luận
  Lần cuộn 13/40: thu được 198 bình luận
Không tải thêm được bình luận mới sau 4 lần cuộn liên tiếp, dừng thu thập.

Tổng số bình luận thu được: 198
URL thực tế sau điều hướng: https://www.tikto

,comment,comment_id,username,nickname,likes,reply_count,created_at,post_url
0,Coi riết t thù luôn chứ không ghét nữa,7708918364299068178,kate.t94,Kate 🦀,2970,21,2026-11-17 06:40:54,https://www.tiktok.com/@phamlocvt/video/770824...
1,Giống điểm gì 😅,7708324279394796306,vivianguyen0707,vivianguyen0707,245,4,2026-11-15 16:15:21,https://www.tiktok.com/@phamlocvt/video/770824...
2,"hổng ai kêu anh là TT hết á, Hai của tui cao r...",7708588441177899777,coconut.100492,G.daily_BT71🌴,445,2,2026-11-16 09:20:42,https://www.tiktok.com/@phamlocvt/video/770824...
3,Kêu anh là trấn thành chi vậy anh,7708711626392617748,hong50594,Hong,21,1,2026-11-16 17:18:31,https://www.tiktok.com/@phamlocvt/video/770824...
4,gần tháng này chưa xem cái nào trùng cái nào l...,7709007740303934216,hongmnhcng331,Mạnhh Cườngg,126,6,2026-11-17 12:27:36,https://www.tiktok.com/@phamlocvt/video/770824...
5,Mình là người luôn giữ lập trường ai nói gì nó...,7709611849318941460,zz88an,Anzz Nguyễn,35,1,2026-11-19 03:31:52,https://www.tiktok.com/@phamlocvt/video/770824...
6,Nhưng Trấn Thành k bị nhầm là trường giang,7709071915969266484,ttth1701,TRAN5207418✅,0,0,2026-11-17 16:36:32,https://www.tiktok.com/@phamlocvt/video/770824...
7,gì má,7708326466639315719,hoasen8854,hoa sen 🪷,14,1,2026-11-15 16:23:41,https://www.tiktok.com/@phamlocvt/video/770824...
8,Ụa rồi mắc gì căng. 🤣,7710487891508790036,quin_1503,quin_1503,1,0,2026-11-21 12:11:25,https://www.tiktok.com/@phamlocvt/video/770824...
9,Dị đó mà ảnh lập vi bằng đó,7708963104156173062,manhkhoimayxaydung,Lynhh.lynhh,3,0,2026-11-17 09:34:46,https://www.tiktok.com/@phamlocvt/video/770824...


In [7]:
# === XUẤT RA FILE CSV ĐỂ DÙNG CHO phase1.ipynb ===
if df_comments.empty:
    print("[Cảnh báo] Chưa có bình luận nào để xuất file CSV.")
else:
    post_id = extract_post_id(resolved_url)
    output_path = Path(OUTPUT_CSV) if OUTPUT_CSV else Path("data") / f"tiktok_comments_{post_id}.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Sắp xếp cột `comment` đầu tiên để phase1.ipynb và src.pipeline đọc tự động
    preferred_cols = [
        "comment",
        "comment_id",
        "username",
        "nickname",
        "likes",
        "reply_count",
        "created_at",
        "post_url",
    ]
    ordered_cols = [c for c in preferred_cols if c in df_comments.columns]
    df_export = df_comments[ordered_cols]
    df_export.to_csv(output_path, index=False, encoding="utf-8-sig")

    rel_path = output_path.as_posix()
    print(f"Đã xuất {len(df_export)} bình luận ra file: {output_path.resolve()}")
    print("\nCopy dòng dưới đây dán vào cell cấu hình của `phase1.ipynb`:")
    print(f'COMMENTS_PATH = "{rel_path}"')

Đã xuất 198 bình luận ra file: E:\A\Sentiment-Analysis\data\tiktok_comments_7708245390318996737.csv

Copy dòng dưới đây dán vào cell cấu hình của `phase1.ipynb`:
COMMENTS_PATH = "data/tiktok_comments_7708245390318996737.csv"
